# 01 · ASVspoof 5 — preparazione dati

Per ciascuno split (**train** → addestramento, **dev** → selezione del modello, **eval** → test finale):

1. download ed estrazione;
2. esplorazione del protocollo;
3. selezione di un sottoinsieme bilanciato bonafide/spoof (salvato come **manifest** in `manifests/`, da committare);
4. estrazione dei Mel-spettrogrammi in `ARTIFACTS_ROOT/features/asvspoof5/<config>/`.

Gli attacchi sono disgiunti tra gli split (train A01–A08, dev A09–A16, eval A17–A32 + codec C01–C11):
l'eval misura la generalizzazione ad attacchi mai visti.

Funziona sia in locale sia su Colab. Il codice riusabile sta in `src/thesis/`; qui ci sono solo parametri, esecuzione e analisi.

## 0 · Setup

- **Locale**: una volta sola, dalla radice del repo: `pip install -e ".[notebook]"`.
- **Colab**: la cella sotto monta Drive, clona il repo e installa le dipendenze.

Per spostare i dati altrove imposta `THESIS_DATA_ROOT` / `THESIS_ARTIFACTS_ROOT` **prima** di importare `thesis`.
Su Colab ASVspoof 5 non sta nel Drive gratuito: usa ad es. `/content/data` (temporaneo).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/SB2526-MasterThesis")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/espositic/SB2526-MasterThesis", str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[notebook]"], check=True)
    # os.environ["THESIS_DATA_ROOT"] = "/content/data"
else:
    REPO = Path.cwd().resolve().parent  # il notebook gira da notebooks/

# Rende importabile `thesis` senza dover riavviare il kernel dopo l'installazione
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

In [ ]:
import logging

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from thesis import config
from thesis.datasets import ASVspoof5
from thesis.datasets.base import audio_duration
from thesis.features import MelConfig, extract_mel, features_dir

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s - %(message)s", force=True)
for noisy in ("httpx", "httpx2", "huggingface_hub"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
print(config.describe())

## 1 · Parametri dell'esperimento

In [ ]:
# Campioni per classe (bonafide e spoof) in ciascuno split
SUBSETS = {
    "train": 5_000,
    "dev":   2_500,
    "eval":  5_000,
}
MIN_DURATION = 1.0       # secondi
MAX_DURATION = 15.0      # secondi
SEED = config.DEFAULT_SEED

MEL = MelConfig(sample_rate=16_000, n_mels=128, n_fft=1024, hop_length=512)
MAX_WORKERS = None       # None = tutti i core

# True = cancella ogni TAR subito dopo l'estrazione (dev+eval: ~105 GB invece di ~210 GB)
DELETE_TARS = True

## 2 · Download ed estrazione

Per ogni split scarica da Hugging Face i TAR **uno alla volta** (download → estrazione → file successivo).
Si può interrompere e rilanciare: i TAR già estratti vengono saltati, gli split completi pure.
Dimensioni: train ~37 GB, dev ~20 GB, eval ~85 GB.

In [ ]:
ds = ASVspoof5()
for split in SUBSETS:
    ds.prepare(split, delete_tars=DELETE_TARS)

## 3 · Esplorazione dei protocolli

In [ ]:
protocols = {split: ds.load_protocol(split) for split in SUBSETS}

summary = pd.DataFrame({
    split: {
        "bonafide": (p["label"] == "bonafide").sum(),
        "spoof": (p["label"] == "spoof").sum(),
        "speaker": p["speaker_id"].nunique(),
        "attacchi": ", ".join(sorted(p.loc[p["label"] == "spoof", "attack_label"].unique())),
    }
    for split, p in protocols.items()
}).T
display(summary)

In [ ]:
fig, axes = plt.subplots(1, len(protocols), figsize=(5 * len(protocols), 3.5))
for ax, (split, p) in zip(np.atleast_1d(axes), protocols.items()):
    p[p["label"] == "spoof"]["attack_label"].value_counts().sort_index().plot.bar(ax=ax, title=f"Attacchi — {split}")
plt.tight_layout()

eval_codecs = protocols.get("eval", pd.DataFrame(columns=["codec"]))["codec"].value_counts().sort_index()
if len(eval_codecs):
    eval_codecs.plot.bar(figsize=(8, 3), title="Codec — eval ('-' = nessuno)")

## 4 · Sottoinsiemi bilanciati

Per ogni split, se il manifest per questa configurazione esiste viene ricaricato (e deve trovare **tutti** i file su disco),
altrimenti viene generato e salvato in `manifests/`.

In [ ]:
subsets, manifests = {}, {}
for split, n in SUBSETS.items():
    subsets[split] = ds.balanced_subset(split, n, MIN_DURATION, MAX_DURATION, SEED)
    manifests[split] = ds.manifest_path(split, n, SEED, MIN_DURATION, MAX_DURATION)
    print(f"{split:5s} → {manifests[split].name}: {subsets[split]['label'].value_counts().to_dict()}")

In [ ]:
for split, subset in subsets.items():
    # I manifest migrati dal vecchio progetto non hanno la durata: la si ricava dall'header.
    missing = subset["duration"].isna()
    if missing.any():
        subset.loc[missing, "duration"] = [audio_duration(ds.abs_path(p)) for p in subset.loc[missing, "audio_path"]]

# Metadati del protocollo (attacco, codec, speaker) per i campioni selezionati
subsets_meta = {
    split: subset.merge(protocols[split].drop(columns=["label", "audio_path"]), on="file_name", how="left")
    for split, subset in subsets.items()
}

fig, axes = plt.subplots(2, len(subsets_meta), figsize=(5 * len(subsets_meta), 7), squeeze=False)
for col, (split, meta) in enumerate(subsets_meta.items()):
    for label, g in meta.groupby("label"):
        axes[0, col].hist(g["duration"], bins=50, alpha=0.6, label=label)
    axes[0, col].set(title=f"Durata (s) — {split}"); axes[0, col].legend()
    meta[meta["label"] == "spoof"]["attack_label"].value_counts().sort_index().plot.bar(ax=axes[1, col], title=f"Attacchi nel subset — {split}")
plt.tight_layout()

display(pd.DataFrame({split: {"speaker distinti": m["speaker_id"].nunique(), "durata media (s)": round(m["duration"].mean(), 2)}
                      for split, m in subsets_meta.items()}).T)

## 5 · Estrazione Mel-spettrogrammi

I file `.npy` già presenti vengono riusati. Per ogni split viene scritto `index_<manifest>.csv`
(label 0 = bonafide, 1 = spoof): è il punto di partenza per i notebook di training.

In [ ]:
FEAT_DIR = features_dir(ds, MEL)
indexes = {
    split: extract_mel(ds, subset, subset_name=manifests[split].stem, config=MEL, max_workers=MAX_WORKERS)
    for split, subset in subsets.items()
}
print(FEAT_DIR)
for split, index in indexes.items():
    print(f"{split:5s}: {len(index):,} spettrogrammi")

## 6 · Controlli

In [ ]:
display(pd.DataFrame({split: idx["n_frames"].describe() for split, idx in indexes.items()}).T.round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5), gridspec_kw={"width_ratios": [1, 1.4, 1.4]})
for split, idx in indexes.items():
    axes[0].hist(idx["n_frames"], bins=50, alpha=0.5, label=split)
axes[0].set(title="Lunghezza spettrogrammi", xlabel="frame"); axes[0].legend()
train_index = indexes["train"]
for ax, (label, name) in zip(axes[1:], [(0, "bonafide"), (1, "spoof")]):
    row = train_index[train_index["label"] == label].iloc[0]
    mel = np.load(FEAT_DIR / row["feature_path"])
    im = ax.imshow(mel, origin="lower", aspect="auto", cmap="magma")
    ax.set(title=f"{name} — {row['file_name']}", xlabel="frame", ylabel="mel")
    fig.colorbar(im, ax=ax, format="%+.0f dB")
plt.tight_layout()